## 1. Initialize Project Environment
Import libraries for correlation computation and adjacency matrix construction.

In [10]:
from __future__ import annotations

import logging
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Optional, Tuple

import numpy as np
import pandas as pd
from scipy import stats

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")

print("pandas", pd.__version__)
print("numpy", np.__version__)

pandas 2.2.3
numpy 2.1.3


## 2. Define Configuration Parameters
Centralize correlation method, thresholding strategy, and soft-thresholding power for WGCNA-style networks.

In [11]:
@dataclass
class NetworkConfig:
    input_file: Path = Path("artifacts/task1_expression_preprocessed.csv")
    export_dir: Path = Path("artifacts")
    corr_method: str = "pearson"
    use_abs_correlation: bool = True
    hard_threshold: Optional[float] = None
    soft_power: int = 6
    use_soft_threshold: bool = True

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        info["input_file"] = str(info["input_file"])
        info["export_dir"] = str(info["export_dir"])
        return info


CONFIG = NetworkConfig()
CONFIG.describe()

{'input_file': 'artifacts/task1_expression_preprocessed.csv',
 'export_dir': 'artifacts',
 'corr_method': 'pearson',
 'use_abs_correlation': True,
 'hard_threshold': None,
 'soft_power': 6,
 'use_soft_threshold': True}

## 3. Load Preprocessed Expression Data

In [12]:
def load_expression(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, index_col=0)
    logging.info(
        f"Loaded expression matrix: {df.shape[0]} genes x {df.shape[1]} samples"
    )
    return df


expr_data = load_expression(CONFIG.input_file)
expr_data.head()

[INFO] Loaded expression matrix: 67 genes x 30 samples


,Sample_1,Sample_2,Sample_3,Sample_4,Sample_5,Sample_6,Sample_7,Sample_8,Sample_9,Sample_10,...,Sample_21,Sample_22,Sample_23,Sample_24,Sample_25,Sample_26,Sample_27,Sample_28,Sample_29,Sample_30
TP53,7.386674,7.281736,7.331440,7.552193,7.258825,7.160664,7.680513,7.451144,7.061801,7.428960,...,6.087345,5.438297,5.785204,5.207313,5.629532,6.018824,5.239168,6.230274,5.434585,6.280421
MDM2,7.333911,7.392634,7.446288,7.570753,7.277110,7.355360,7.555279,7.524734,7.340710,7.396263,...,6.519472,6.093933,5.712958,5.740905,5.952236,5.824134,5.548742,6.113489,5.707637,6.030646
BAX,7.454691,7.395490,7.509959,7.620249,7.247393,7.345950,7.520701,7.432625,7.185235,7.421639,...,6.218403,5.871779,5.936592,4.995132,6.190874,6.082840,4.476583,6.072094,5.464406,5.998384
PUMA,7.289786,7.393231,7.356408,7.504150,7.255060,7.060195,7.609524,7.509945,7.215700,7.504806,...,5.814904,5.242988,6.143685,5.527324,5.485711,5.679927,5.231850,6.381528,5.623482,5.882170
NOXA,7.412487,7.225669,7.427038,7.531754,7.275123,7.325341,7.685693,7.334594,7.412622,7.226611,...,6.482543,5.861129,6.078565,5.438473,5.391791,5.768248,5.604320,6.132978,5.621224,5.787595


## 4. Compute Correlation Matrix

In [13]:
def compute_correlation_matrix(
    df: pd.DataFrame, method: str = "pearson", use_abs: bool = True
) -> pd.DataFrame:
    corr = df.T.corr(method=method)
    if use_abs:
        corr = corr.abs()
        logging.info(f"Using absolute {method} correlation (unsigned network)")
    np.fill_diagonal(corr.values, 1.0)
    logging.info(f"Correlation matrix shape: {corr.shape}")
    return corr


corr_matrix = compute_correlation_matrix(
    expr_data, method=CONFIG.corr_method, use_abs=CONFIG.use_abs_correlation
)
corr_matrix.iloc[:6, :6]

[INFO] Using absolute pearson correlation (unsigned network)
[INFO] Correlation matrix shape: (67, 67)


,TP53,MDM2,BAX,PUMA,NOXA,BID
TP53,1.000000,0.957172,0.945312,0.963905,0.966270,0.940805
MDM2,0.957172,1.000000,0.952888,0.950234,0.969623,0.946679
BAX,0.945312,0.952888,1.000000,0.936650,0.923272,0.901072
PUMA,0.963905,0.950234,0.936650,1.000000,0.949504,0.919590
NOXA,0.966270,0.969623,0.923272,0.949504,1.000000,0.959606
BID,0.940805,0.946679,0.901072,0.919590,0.959606,1.000000


In [14]:
corr_values = corr_matrix.values[np.triu_indices(len(corr_matrix), k=1)]
corr_stats = {
    "n_pairs": len(corr_values),
    "mean": corr_values.mean(),
    "std": corr_values.std(),
}
pd.DataFrame([corr_stats])

,n_pairs,mean,std
0,2211,0.660071,0.265379


## 5. Construct Adjacency Matrix

In [15]:
def soft_threshold_adjacency(corr: pd.DataFrame, power: int = 6) -> pd.DataFrame:
    adj = corr**power
    np.fill_diagonal(adj.values, 0)
    adj_vals = adj.values[np.triu_indices(len(adj), k=1)]
    logging.info(
        f"Soft-threshold adjacency (power={power}): mean={adj_vals.mean():.6f}"
    )
    return adj


def hard_threshold_adjacency(
    corr: pd.DataFrame, threshold: float = 0.7
) -> pd.DataFrame:
    adj = (corr >= threshold).astype(int)
    np.fill_diagonal(adj.values, 0)
    n_edges = (adj.values > 0).sum() // 2
    logging.info(f"Hard-threshold adjacency (threshold={threshold}): {n_edges} edges")
    return adj


if CONFIG.use_soft_threshold:
    adjacency = soft_threshold_adjacency(corr_matrix, CONFIG.soft_power)
    threshold_method = f"soft (power={CONFIG.soft_power})"
else:
    adjacency = hard_threshold_adjacency(corr_matrix, CONFIG.hard_threshold or 0.7)
    threshold_method = f"hard (threshold={CONFIG.hard_threshold or 0.7})"

adjacency.iloc[:6, :6]

[INFO] Soft-threshold adjacency (power=6): mean=0.261228


,TP53,MDM2,BAX,PUMA,NOXA,BID
TP53,0.000000,0.769024,0.713592,0.802058,0.813936,0.693423
MDM2,0.769024,0.000000,0.748603,0.736180,0.831033,0.719807
BAX,0.713592,0.748603,0.000000,0.675248,0.619410,0.535250
PUMA,0.802058,0.736180,0.675248,0.000000,0.732794,0.604737
NOXA,0.813936,0.831033,0.619410,0.732794,0.000000,0.780834
BID,0.693423,0.719807,0.535250,0.604737,0.780834,0.000000


In [16]:
def pick_soft_threshold(corr: pd.DataFrame, powers: list = None) -> pd.DataFrame:
    if powers is None:
        powers = [2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 14, 16]
    results = []
    for power in powers:
        adj = corr**power
        np.fill_diagonal(adj.values, 0)
        connectivity = adj.sum(axis=1)
        mean_conn = connectivity.mean()
        results.append({"power": power, "mean_connectivity": mean_conn})
    return pd.DataFrame(results)


power_analysis = pick_soft_threshold(corr_matrix)
power_analysis

,power,mean_connectivity
0,2,33.403847
1,3,27.163366
2,4,22.851620
3,5,19.678890
4,6,17.241062
5,7,15.303528
6,8,13.720723
7,9,12.398449
8,10,11.273697
9,12,9.455617


## 6. Validate with Unit Tests

In [17]:
def test_correlation_symmetry():
    assert np.allclose(corr_matrix.values, corr_matrix.values.T), (
        "Correlation matrix not symmetric"
    )


def test_correlation_range():
    assert corr_matrix.values.min() >= 0, "Negative correlation in absolute matrix"
    assert corr_matrix.values.max() <= 1, "Correlation > 1"


def test_adjacency_no_self_loops():
    assert np.allclose(np.diag(adjacency.values), 0), "Self-loops present"


def test_soft_threshold():
    test_corr = pd.DataFrame(
        [[1.0, 0.8], [0.8, 1.0]], index=["A", "B"], columns=["A", "B"]
    )
    test_adj = soft_threshold_adjacency(test_corr, power=2)
    assert np.isclose(test_adj.loc["A", "B"], 0.64), (
        f"Expected ~0.64, got {test_adj.loc['A', 'B']}"
    )


test_correlation_symmetry()
test_correlation_range()
test_adjacency_no_self_loops()
test_soft_threshold()
print("All network construction tests passed.")

[INFO] Soft-threshold adjacency (power=2): mean=0.640000


All network construction tests passed.


## 7. Export Results

In [18]:
EXPORT_DIR = CONFIG.export_dir
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

corr_matrix.to_csv(EXPORT_DIR / "task2_correlation_matrix.csv")
print(
    f"[OK] Correlation matrix saved to: {EXPORT_DIR / 'task2_correlation_matrix.csv'}"
)

adjacency.to_csv(EXPORT_DIR / "task2_adjacency_matrix.csv")
print(f"[OK] Adjacency matrix saved to: {EXPORT_DIR / 'task2_adjacency_matrix.csv'}")

power_analysis.to_csv(EXPORT_DIR / "task2_power_analysis.csv", index=False)
print(f"[OK] Power analysis saved to: {EXPORT_DIR / 'task2_power_analysis.csv'}")

params = {
    "correlation_method": CONFIG.corr_method,
    "threshold_method": threshold_method,
    "n_genes": len(corr_matrix),
}
pd.DataFrame([params]).to_csv(EXPORT_DIR / "task2_network_params.csv", index=False)
print(f"[OK] Network parameters saved to: {EXPORT_DIR / 'task2_network_params.csv'}")

[OK] Correlation matrix saved to: artifacts/task2_correlation_matrix.csv
[OK] Adjacency matrix saved to: artifacts/task2_adjacency_matrix.csv
[OK] Power analysis saved to: artifacts/task2_power_analysis.csv
[OK] Network parameters saved to: artifacts/task2_network_params.csv
